# Fibromyalgia RAG Pipeline

A Retrieval-Augmented Generation (RAG) pipeline built over a single biomedical
review article: *"Fibromyalgia: A Review of the Pathophysiological Mechanisms
and Multidisciplinary Treatment Strategies"* (Jurado-Priego et al., 2024,
*Biomedicines* 12, 1543).

**Pipeline stages:**

1. Setup
2. Data Loading (layout-aware PDF parsing)
3. Heading Detection & Section Building
4. Reference-List Parsing
5. Cleaning
6. Chunking
7. Embedding & Indexing
8. Retrieval Evaluation
9. Answer Generation (RAG)
10. Results

Parsing is layout-aware: headings are detected from the PDF's font metadata
(bold/italic + numbering pattern), not matched against a fixed list of
section-title strings. This means the pipeline recovers subsections (e.g.
"6.1. Pharmacological Treatment") in addition to the 7 top-level sections,
and chunks are grouped per subsection with token-based sizing so retrieval
returns tightly scoped passages with accurate page numbers.


## 1. Setup

In [ ]:
%pip install -q pymupdf langchain-core langchain-text-splitters \
    langchain-community langchain-huggingface faiss-cpu sentence-transformers \
    openai tiktoken


In [ ]:
import os
import re
import sys
from pathlib import Path

# Make the project's `src/` package importable so the notebook can reuse the
# exact same pipeline code as the Gradio app (src/app.py), instead of
# duplicating logic here.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pipeline as P

DATA_RAW_DIR = P.DATA_RAW_DIR
DATA_PROCESSED_DIR = P.DATA_PROCESSED_DIR
PDF_PATH = P.PDF_PATH

assert PDF_PATH.exists(), f"Source PDF not found at {PDF_PATH}."
print("Using PDF:", PDF_PATH)


## 2. Data Loading (layout-aware extraction)

The PDF is parsed with PyMuPDF using the `"dict"` output, which provides
font/formatting metadata for every text line in addition to its content and
position. Image blocks are excluded, since this pipeline focuses on textual
content. Repeated running headers/footers and bare page-number lines (e.g.
`"4 of 22"`) are filtered out at this stage via `JUNK_LINE_PATTERNS`.


In [ ]:
lines = P.extract_lines(PDF_PATH)
print(f"Extracted {len(lines)} text lines (after removing running headers/footers)")
lines[5]


## 3. Heading Detection & Section Building

Headings are identified from their numbering pattern *and* formatting:

| Level | Pattern | Formatting |
|---|---|---|
| 1 | `N. Title` | Bold |
| 2 | `N.N. Title` | Italic |
| 3 | `N.N.N. Title` | Regular text on its own line |

The line stream is then sliced between consecutive headings into
paragraph-level elements, each carrying **page number, section, and
subsection** metadata. Hyphenated words split across a line break are
rejoined at this point.


In [ ]:
headings = P.detect_headings(lines)
print(f"Detected {len(headings)} headings\n")
for h in headings:
    print(" " * ((h["level"] - 1) * 3), h["number"], h["title"])


In [ ]:
parsed_elements = P.build_sections(lines, headings)
for e in parsed_elements[:3]:
    print(f"[{e['page_number']}] {e['section']} -> {e['subsection']}: {e['text'][:60]}...")
print(f"\nTotal elements (paragraphs): {len(parsed_elements)}")


## 4. Reference-List Parsing

Parsed into individually addressable, numbered entries so an in-text marker
like `[12,45]` can be resolved back to its source. This is not required by
Cleaning or Chunking below, but is preserved as a standalone utility (e.g.
for citation lookups).


In [ ]:
references = P.parse_references(PDF_PATH)
print(f"Parsed {len(references)} reference entries")
if references:
    first_key = sorted(references)[0]
    print(f"Entry {first_key}: {references[first_key][:100]}...")


## 5. Cleaning

In [ ]:
cleaned_elements = P.clean_elements(parsed_elements)
print(f"Cleaned elements: {len(cleaned_elements)} (from {len(parsed_elements)} parsed)")
for e in cleaned_elements[:3]:
    print(f"[{e['page_number']}] {e['section']} -> {e['subsection']}: {e['text'][:80]}...")


## 6. Chunking

Elements are grouped by `(section, subsection)`, combined into a continuous
string, and split with `RecursiveCharacterTextSplitter` using a token-based
length function (`tiktoken`, `cl100k_base`) instead of raw character count.
Each chunk's page number(s) are recovered via a character-offset -> page map
built while combining the elements, so page attribution stays accurate even
though the chunk boundaries no longer line up 1:1 with the original elements.

Chunks below a minimal token count or with no meaningful alphabetic content
(pure page-number/punctuation noise) are filtered out.


In [ ]:
final_chunks = P.chunk_elements(cleaned_elements)
print(f"Generated {len(final_chunks)} chunks.")

filtered_chunks = P.filter_meaningful_chunks(final_chunks)
print(f"Final chunk count after filtering: {len(filtered_chunks)}")

# Regression check: cleaning must have removed page-number artifacts, and
# chunking must not have reintroduced any.
bad_chunks = [c for c in filtered_chunks if re.search(r"\b\d+\s+of\s+22\b", c["text"])]
assert not bad_chunks, "Text cleaning regression: page numbers leaked into chunks"


In [ ]:
for c in filtered_chunks[:3]:
    print("=" * 80)
    print(f"CHUNK {c['chunk_id']} | {c['section']} -> {c['subsection']} | pages={c['page_numbers']} | tokens={c['n_tokens']}")
    print(c["text"][:300])


## 7. Embedding & Indexing

Chunks are converted to LangChain `Document`s and embedded, then indexed in
a FAISS vector store for similarity search.

Two embedding backends are supported (selected via the `EMBEDDING_BACKEND`
environment variable, read once at import time in `src/pipeline.py`):

- **`huggingface`** (default) -- a local `sentence-transformers` model
  (`all-MiniLM-L6-v2`). No API key required.
- **`openrouter`** -- OpenRouter's embeddings API. Requires the
  `OPENROUTER_API_KEY` environment variable to be set; the key is only ever
  read from the environment and is never hardcoded or committed.

```bash
export EMBEDDING_BACKEND=openrouter
export OPENROUTER_API_KEY="sk-or-..."
```

This notebook uses whichever backend is configured in the environment
(default: `huggingface`), via `src/pipeline.py`'s `get_retriever()`.


In [ ]:
documents = P.build_documents(filtered_chunks)
print(f"Built {len(documents)} chunk-level documents")
documents[0]


In [ ]:
print(f"Embedding backend: {P.EMBEDDING_BACKEND}")

vectorstore, embeddings = P.build_vectorstore(documents)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Indexing complete:", vectorstore.index.ntotal, "vectors")

# Persist the index so it doesn't need to be rebuilt on every run
P.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(P.INDEX_DIR))
print("Saved FAISS index to", P.INDEX_DIR)


## 8. Retrieval Evaluation

A lightweight Precision@K benchmark: for each test question, check whether
any of the top-K retrieved chunks contain at least one of the expected
keywords.


In [ ]:
eval_dataset = [
    {
        "question": "What are the FDA-approved drugs for fibromyalgia?",
        "keywords": ["pregabalin", "duloxetine", "milnacipran"],
    },
    {
        "question": "What is fibromyalgia characterized by?",
        "keywords": ["chronic", "widespread", "pain"],
    },
    {
        "question": "What diagnostic tools or criteria are mentioned?",
        "keywords": ["WPI", "SS scale", "ACR"],
    },
]

precision_at_k = P.evaluate_retrieval(eval_dataset, retriever)
print(f"Retrieval Precision@K Score: {precision_at_k:.2f}%")


In [ ]:
query = "What are the FDA-approved drugs for fibromyalgia?"
results = retriever.invoke(query)

print(f"Query: {query}\n" + "=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n[Chunk {i}] {doc.metadata['section']} -> {doc.metadata['subsection']} (pages {doc.metadata['page_numbers']})")
    print(doc.page_content.strip())
    print("-" * 60)


## 9. Answer Generation (RAG)

If an `OPENAI_API_KEY` environment variable is set, the retrieved chunks are
passed to the OpenAI Chat Completions API to produce a grounded answer.
Without a key, this falls back to a template-based extractive answer built
directly from the retrieved chunks, so the notebook remains fully runnable
without any secret.


In [ ]:
answer, docs = P.generate_answer(
    "What non-pharmacological treatments are discussed for fibromyalgia?",
    retriever,
)
print(answer)


## 10. Results

- The article is parsed layout-aware into its top-level sections *and*
  numbered subsections (via bold/italic heading detection), rather than
  matched against a fixed list of 7 section-title strings.
- Chunking is grouped per subsection and sized by token count (`tiktoken`),
  with each chunk's page number(s) recovered from a character-offset -> page
  map, and a regression assertion confirms no leftover `"N of 22"`
  page-number artifacts survive into the final chunks.
- Small, low-signal chunks (fewer than 5 tokens, or no meaningful
  alphabetic content) are filtered out before embedding.
- The reference list is parsed separately into numbered, addressable
  entries (available for citation lookups, not required by the pipeline
  above).
- A 3-question Precision@K retrieval benchmark is included as a smoke test;
  the exact score is printed above and depends on the embedding model's
  live output, so it is not hard-coded here.
- The RAG loop is complete end-to-end: retrieval feeds into an answer
  generation step (OpenAI when `OPENAI_API_KEY` is set, extractive fallback
  otherwise).

## Limitations

- The corpus is a single article; the retrieval evaluation is a 3-question
  smoke test, not a statistically powered benchmark.
- Heading detection depends on the source PDF's font metadata (bold/italic
  spans and numbering pattern) and would need adapting for articles that
  format headings differently.
- Full LLM-based generation requires an OpenAI API key (not included, and
  never committed to this repository). Without one, generation degrades to
  an extractive summary of the retrieved context.
- The OpenRouter embedding backend requires `OPENROUTER_API_KEY` to be set
  in the environment; it is never hardcoded or committed anywhere in this
  project.
- No automated evaluation (e.g. faithfulness or answer-relevance scoring) is
  performed on the generation step itself -- only on retrieval.
